<a href="https://colab.research.google.com/github/Kunsh1/AutoEIT/blob/main/AutoEIT_Test_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoEIT — Test I: Audio-to-Text Transcription
## GSoC 2026 | HumanAI Foundation (NIU + University of Alabama)

---

## Background

The **Elicited Imitation Task (EIT)** is a validated tool for measuring global oral proficiency in second language acquisition (SLA) research (Ortega et al., 2002; Faretta-Stutenberg et al., 2023). Participants listen to 30 sentences of increasing complexity and **repeat as much as they can**. Responses are transcribed and scored 0–4 based on meaning preservation.

**Current bottleneck:** Two independent human raters must manually transcribe each recording before scoring. This is slow, expensive, and hard to scale.

**This notebook:** Implements and evaluates an automated transcription pipeline for the 4 sample EIT-A audio files provided in Test I.

---

### EIT Scoring Rubric (Ortega, 2000)

| Score | Criteria | Example |
|:---:|---|---|
| **4** | Exact repetition — form and meaning both correct | Perfect repeat |
| **3** | Meaning preserved; ungrammatical changes OK if meaning unchanged | `El chico con el salgo es español` |
| **2** | >50% idea units; meaning close but inexact/incomplete/ambiguous | `Ella sola cerveza y no come nada` |
| **1** | ≤50% idea units; important info missing or opposed to stimulus | `Antes de poder seguir... perdió su cuarto` |
| **0** | Silence, garbled output, or only 1 content word | `[no response]` / `XXX` |

**Key rules (from protocol):**
- Always score the **best final response** — self-corrections are NOT penalized
- Transcribe ALL disfluencies: `eh`, `mm`, `um`, `este`, false starts (`la- las`), pauses (`...`)
- **Never correct** learner grammar, vocabulary, or pronunciation errors
- Use `[no response]` for silence or unintelligible output

---

### Audio Structure

Each EIT recording has this structure:
```
[~55s English instructions + practice sentences]
     ↓
Repeat × 30:
   [Recorded Spanish sentence plays through mic]  ← DO NOT TRANSCRIBE
   [2–3s delay]
   [TONE BEEP]  ← marks start of participant response window
   [Participant's repetition, ~3–8s]  ← TRANSCRIBE THIS
   [Silence before next sentence]
```

The 4 sample files are all **EIT Version A**:

| File | Participant | Session |
|---|---|---|
| `038010_EIT-2A.mp3` | 038010 | 2 |
| `038011_EIT-1A.mp3` | 038011 | 1 |
| `038012_EIT-2A.mp3` | 038012 | 2 |
| `038015_EIT-1A.mp3` | 038015 | 1 |

---
## Section 1 — Environment Setup

In [ ]:
# Install dependencies (run once, then comment out)
!pip install openai-whisper pydub scipy jiwer pandas openpyxl tqdm librosa soundfile matplotlib -q
!apt-get install -y ffmpeg -q  # required by pydub for mp3 decoding
print('Done.')

In [ ]:
import sys, os, re, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import Audio, display, HTML

import torch
import whisper
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from scipy.signal import butter, filtfilt
import librosa
import librosa.display

warnings.filterwarnings('ignore')

# ── Environment info ──────────────────────────────────────────────────────────
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Whisper : {whisper.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Running on CPU — consider using model="medium" for speed')

---
## Section 2 — Data & Paths

Download the audio files from the [AutoEIT SharePoint folder](https://niuits-my.sharepoint.com/:f:/g/personal/a1757262_mail_niu_edu/IgDqP3Aihu7qSoIqbdLC-bPwAdG-Jh8hs7IPM9XQRWNLPQo?e=IXBjjV) and place them in `./data/audio/`. Also place both Excel files in `./data/`.

```
data/
├── audio/
│   ├── 038010_EIT-2A.mp3
│   ├── 038011_EIT-1A.mp3
│   ├── 038012_EIT-2A.mp3
│   └── 038015_EIT-1A.mp3
├── AutoEIT_Sample_Audio_for_Transcribing.xlsx
└── AutoEIT_Sample_Transcriptions_for_Scoring.xlsx
```

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
AUDIO_DIR    = Path('./data/audio')
TEMPLATE_XLS = './data/AutoEIT_Sample_Audio_for_Transcribing.xlsx'
REFERENCE_XLS= './data/AutoEIT_Sample_Transcriptions_for_Scoring.xlsx'
OUTPUT_DIR   = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Collect audio files
audio_files = []
if AUDIO_DIR.exists():
    for ext in ('*.mp3','*.wav','*.m4a','*.flac'):
        audio_files.extend(AUDIO_DIR.glob(ext))
    audio_files = sorted(set(audio_files))

if audio_files:
    print(f'Found {len(audio_files)} audio file(s):')
    for f in audio_files:
        print(f'  ✅  {f.name:35s}  {f.stat().st_size/1e6:.1f} MB')
else:
    print('⚠️  No audio files found. Place mp3 files in ./data/audio/')

print()
print(f'Template  : {"✅" if Path(TEMPLATE_XLS).exists() else "❌ missing"}  {TEMPLATE_XLS}')
print(f'Reference : {"✅" if Path(REFERENCE_XLS).exists() else "❌ missing"}  {REFERENCE_XLS}')

---
## Section 3 — EIT Version A Stimuli

All 30 target sentences for EIT Version A (Faretta-Stutenberg et al., 2023 / Ortega et al., 2002). These are the sentences participants listened to and attempted to repeat. Sentences increase in length from 7 to 18 syllables.

In [ ]:
EIT_VERSION_A_STIMULI = [
    "Quiero cortarme el pelo",                                        # 1  (7 syl)
    "El libro está en la mesa",                                       # 2  (7)
    "El carro lo tiene Pedro",                                        # 3  (8)
    "Él se ducha cada mañana",                                        # 4  (9)
    "¿Qué dice usted que va a hacer hoy?",                            # 5  (9)
    "Dudo que sepa manejar muy bien",                                 # 6  (10)
    "Las calles de esta ciudad son muy anchas",                       # 7  (11)
    "Puede que llueva mañana todo el día",                            # 8  (12)
    "Las casas son muy bonitas pero caras",                           # 9  (12)
    "Me gustan las películas que acaban bien",                        # 10 (12)
    "El chico con el que yo salgo es español",                        # 11 (13)
    "Después de cenar me fui a dormir tranquilo",                     # 12 (13)
    "Quiero una casa en la que vivan mis animales",                   # 13 (14)
    "A nosotros nos fascinan las fiestas grandiosas",                 # 14 (14)
    "Ella sólo bebe cerveza y no come nada",                          # 15 (15)
    "Me gustaría que el precio de las casas bajara",                  # 16 (15)
    "Cruza a la derecha y después sigue todo recto",                  # 17 (15)
    "Ella ha terminado de pintar su apartamento",                     # 18 (15)
    "Me gustaría que empezara a hacer más calor pronto",              # 19 (15)
    "El niño al que se le murió el gato está triste",                 # 20 (16)
    "Una amiga mía cuida a los niños de mi vecino",                   # 21 (16)
    "El gato que era negro fue perseguido por el perro",              # 22 (16)
    "Antes de poder salir él tiene que limpiar su cuarto",            # 23 (16)
    "La cantidad de personas que fuman ha disminuido",                # 24 (17)
    "Después de llegar a casa del trabajo tomé la cena",              # 25 (17)
    "El ladrón al que atrapó la policía era famoso",                  # 26 (17)
    "Le pedí a un amigo que me ayudara con la tarea",                 # 27 (17)
    "El examen no fue tan difícil como me habían dicho",              # 28 (17)
    "¿Serías tan amable de darme el libro que está en la mesa?",      # 29 (18)
    "Hay mucha gente que no toma nada para el desayuno",              # 30 (18)
]

# Display as table
syllable_counts = [7,7,8,9,9,10,11,12,12,12,13,13,14,14,15,15,15,15,15,16,16,16,16,17,17,17,17,17,18,18]
stim_df = pd.DataFrame({'#': range(1,31), 'Stimulus': EIT_VERSION_A_STIMULI, 'Syllables': syllable_counts})

# Complexity plot
fig, ax = plt.subplots(figsize=(13, 3))
ax.bar(stim_df['#'], stim_df['Syllables'], color='#3498db', edgecolor='none', alpha=0.85)
ax.set_xlabel('Sentence #', fontsize=10)
ax.set_ylabel('Syllables', fontsize=10)
ax.set_title('EIT Version A — Sentence Complexity (7→18 syllables)', fontsize=11)
ax.set_xticks(range(1,31))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'eit_complexity.png', dpi=120, bbox_inches='tight')
plt.show()

print(stim_df.to_string(index=False))

---
## Section 4 — Audio Exploration

Before building the transcription pipeline, we need to understand the audio structure. The recording contains **both** the EIT playback (stimulus sentences) and the participant's voice. We must transcribe only the participant's response window — the portion after each tone beep.

In [ ]:
def load_audio_np(path, sr=16000):
    """Load audio file as mono float32 numpy array at 16kHz."""
    seg = AudioSegment.from_file(str(path)).set_channels(1).set_frame_rate(sr)
    arr = np.array(seg.get_array_of_samples()).astype(np.float32)
    arr /= float(np.iinfo(seg.array_type).max)
    return arr, sr

if audio_files:
    sample_file = audio_files[0]
    y, sr = load_audio_np(sample_file)
    duration = len(y) / sr
    t = np.arange(len(y)) / sr

    print(f'File     : {sample_file.name}')
    print(f'Duration : {duration:.1f}s  ({duration/60:.1f} min)')
    print(f'SR       : {sr} Hz')
    print(f'Samples  : {len(y):,}')
    print(f'Peak amp : {np.abs(y).max():.4f}')

    # ── Plot full waveform + zoomed response section ───────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(16, 9))

    # Full waveform
    axes[0].plot(t, y, lw=0.3, color='#2980b9')
    axes[0].axvspan(0, 55, color='red', alpha=0.08, label='Intro / practice (~55s)')
    axes[0].axvline(55, color='red', lw=1.5, linestyle='--')
    axes[0].set_title(f'Full Waveform — {sample_file.name}  ({duration:.0f}s)', fontsize=11)
    axes[0].set_ylabel('Amplitude')
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].set_xlim(0, duration)

    # Zoom: sentences 1-5 (approx 55–130s)
    z1, z2 = 55, 130
    mask = (t >= z1) & (t <= z2)
    axes[1].plot(t[mask], y[mask], lw=0.5, color='#27ae60')
    axes[1].set_title(f'Zoomed: sentences 1–5 (approx {z1}–{z2}s) — can see stimulus + tone + response pattern', fontsize=10)
    axes[1].set_ylabel('Amplitude')
    axes[1].set_xlim(z1, z2)

    # RMS energy — reveals activity pattern
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    t_rms = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)
    db_rms = librosa.amplitude_to_db(rms, ref=np.max)
    axes[2].plot(t_rms, db_rms, lw=0.6, color='#8e44ad')
    axes[2].axhline(-40, color='orange', lw=1, linestyle='--', label='Silence threshold (-40dBFS)')
    axes[2].axvline(55, color='red', lw=1.5, linestyle='--', label='Intro ends')
    axes[2].set_title('RMS Energy (dB) — used for silence detection', fontsize=10)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('dBFS')
    axes[2].legend(fontsize=9)
    axes[2].set_xlim(0, duration)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'waveform_analysis.png', dpi=130, bbox_inches='tight')
    plt.show()

    # Listen to first response section
    print('\nListening to intro end + first response window (50–75s):')
    display(Audio(y[int(50*sr):int(75*sr)], rate=sr))

---
## Section 5 — Tone Detection

The EIT protocol uses a distinct **tone beep** (typically ~880 Hz) immediately before each participant response. Detecting this tone lets us precisely locate the 30 response windows, avoiding transcription of the stimulus playback.

**Method:** Narrow bandpass filter at ±120Hz around the expected tone frequency → RMS energy → peak detection.

In [ ]:
def detect_tone_positions(audio_np, sr=16000, tone_freq=880):
    """Detect tone beep positions using bandpass filter + RMS energy peaks."""
    nyq  = sr / 2
    low  = max(0.01, (tone_freq - 120) / nyq)
    high = min(0.99, (tone_freq + 120) / nyq)
    try:
        b, a     = butter(4, [low, high], btype='band')
        filtered = filtfilt(b, a, audio_np)
    except Exception:
        filtered = audio_np

    window_s = int(0.05 * sr)
    hop_s    = int(0.025 * sr)
    rms = [np.sqrt(np.mean(filtered[i:i+window_s]**2))
           for i in range(0, len(filtered) - window_s, hop_s)]
    rms = np.array(rms)

    thresh = np.mean(rms) + 2.5 * np.std(rms)
    positions, in_tone, tone_start = [], False, 0

    for i, val in enumerate(rms):
        sp = i * hop_s
        if val > thresh and not in_tone:
            in_tone, tone_start = True, sp
        elif val <= thresh and in_tone:
            in_tone = False
            dur = (sp - tone_start) / sr
            if 0.05 < dur < 1.5:
                positions.append(tone_start)
    return positions


if audio_files:
    INTRO_SKIP = 55   # seconds
    y, sr       = load_audio_np(audio_files[0])
    intro_samp  = int(INTRO_SKIP * sr)
    body        = y[intro_samp:]

    # Test multiple frequencies to find the right one
    print('Scanning tone frequencies:')
    for freq in [440, 660, 880, 1000, 1200]:
        tones = detect_tone_positions(body, sr, tone_freq=freq)
        print(f'  {freq} Hz → {len(tones)} tones detected')

    # Use best frequency
    TONE_FREQ = 880
    tones = detect_tone_positions(body, sr, tone_freq=TONE_FREQ)
    tone_times = [intro_samp/sr + t/sr for t in tones]

    print(f'\nUsing {TONE_FREQ} Hz → {len(tones)} tones found (expected 30)')

    # Visualize detected tones on waveform
    fig, ax = plt.subplots(figsize=(16, 3))
    t_body = np.arange(len(y)) / sr
    ax.plot(t_body, y, lw=0.3, color='#7f8c8d', label='Waveform')
    for i, tt in enumerate(tone_times[:30]):
        ax.axvline(tt, color='#e74c3c', lw=0.8, alpha=0.7)
        if i < 5:
            ax.text(tt+0.2, 0.6, str(i+1), fontsize=7, color='#e74c3c')
    ax.axvline(tone_times[0] if tone_times else 55, color='#e74c3c', lw=0.8, label='Detected tone')
    ax.set_title(f'Tone Positions Detected ({len(tones)}) — {audio_files[0].name}', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(INTRO_SKIP - 2, min(t_body[-1], INTRO_SKIP + 200))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'tone_detection.png', dpi=130, bbox_inches='tight')
    plt.show()

---
## Section 6 — Response Window Extraction

After locating each tone, we extract the participant's response window: the audio from ~350ms after each tone beep (to skip the beep itself), up to a maximum of 9 seconds. We then trim trailing silence.

In [ ]:
def find_response_windows(audio_np, sr=16000, intro_skip_s=55, n_expected=30,
                          tone_freq=880, resp_max_s=9.0):
    """Locate 30 participant response windows. Returns list of (start, end) sample tuples."""
    intro_s  = int(intro_skip_s * sr)
    body     = audio_np[intro_s:]
    tones    = sorted(detect_tone_positions(body, sr, tone_freq))

    if len(tones) >= int(n_expected * 0.7):
        tones   = tones[:n_expected]
        windows = []
        for tp in tones:
            start = intro_s + tp + int(0.35 * sr)
            end   = min(start + int(resp_max_s * sr), len(audio_np))
            windows.append((start, end))
        if len(windows) == n_expected:
            return windows

    # Silence-based fallback
    print(f'  ⚠️  Tone detection found {len(tones)} — using silence fallback')
    pcm = (body * 32767).astype(np.int16)
    seg = AudioSegment(pcm.tobytes(), frame_rate=sr, sample_width=2, channels=1)
    ns  = detect_nonsilent(seg, min_silence_len=700, silence_thresh=-40)
    if ns:
        step = max(1, len(ns) // n_expected)
        windows = []
        for i in range(0, len(ns), step):
            s_ms, e_ms = ns[i]
            mid   = (s_ms + e_ms) // 2
            start = intro_s + int(mid / 1000 * sr)
            end   = min(intro_s + int(e_ms / 1000 * sr) + int(3 * sr), len(audio_np))
            windows.append((start, end))
            if len(windows) == n_expected:
                break
        if len(windows) == n_expected:
            return windows

    # Equal-split last resort
    body_len = len(audio_np) - intro_s
    chunk    = body_len // n_expected
    return [(intro_s + i*chunk, min(intro_s + (i+1)*chunk + sr, len(audio_np)))
            for i in range(n_expected)]


if audio_files:
    windows = find_response_windows(y, sr, intro_skip_s=55)
    durations = [(e - s) / sr for s, e in windows]

    print(f'Extracted {len(windows)} response windows')
    print(f'Duration: mean={np.mean(durations):.2f}s  '
          f'min={min(durations):.2f}s  '
          f'max={max(durations):.2f}s')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))

    # Bar chart of window durations
    colors = ['#e74c3c' if d < 1.5 else '#3498db' for d in durations]
    ax1.bar(range(1, 31), durations, color=colors, edgecolor='none')
    ax1.axhline(np.mean(durations), color='black', lw=1.2, linestyle='--',
                label=f'Mean {np.mean(durations):.1f}s')
    ax1.set_xlabel('Sentence #')
    ax1.set_ylabel('Duration (s)')
    ax1.set_title('Response Window Durations')
    ax1.legend()
    short_patch = mpatches.Patch(color='#e74c3c', label='< 1.5s (likely no response)')
    norm_patch  = mpatches.Patch(color='#3498db', label='Normal response')
    ax1.legend(handles=[short_patch, norm_patch], fontsize=8)

    # Histogram
    ax2.hist(durations, bins=15, color='#3498db', edgecolor='white', alpha=0.85)
    ax2.set_xlabel('Duration (s)')
    ax2.set_ylabel('Count')
    ax2.set_title('Duration Distribution')

    plt.suptitle(f'Response Windows — {audio_files[0].name}', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'response_windows.png', dpi=130, bbox_inches='tight')
    plt.show()

    # Listen to first 3 extracted responses
    for i in range(min(3, len(windows))):
        s, e = windows[i]
        print(f'\nSentence {i+1} — stimulus: "{EIT_VERSION_A_STIMULI[i]}"')
        display(Audio(y[s:e], rate=sr))

---
## Section 7 — Whisper Model & Prompt Design

**Model choice:** `whisper-large-v3` — best open-source model for non-native/accented speech.

**Prompt engineering:** Each sentence is transcribed with a stimulus-aware `initial_prompt`. This:
1. Tells Whisper to expect L2 learner speech
2. Provides the target sentence as vocabulary context
3. Instructs Whisper to keep disfluencies (not clean them)
4. Critically: does NOT tell Whisper to reproduce the target — we want what the learner actually said

In [ ]:
def build_prompt(stimulus):
    """Stimulus-aware Whisper prompt for EIT response transcription."""
    return (
        "Transcripción literal de estudiante de español L2. "
        "Conservar todos los errores, disfluencias (eh, mm, um, este), "
        "falsos inicios marcados con guión (ej: 'la- las'), pausas con '...'. "
        f"El estudiante intentó repetir: '{stimulus}'. "
        "Lo que dijo exactamente:"
    )


def format_transcription(raw):
    """
    Apply EIT protocol transcription conventions.
    ONLY removes ASR hallucination artifacts — NEVER fixes learner errors.
    """
    for pat in [r'\[Música\]', r'\[música\]', r'\(música\)',
                r'Subtítulos\s+\S+', r'www\.\S+', r'\[inaudible\]']:
        raw = re.sub(pat, '', raw, flags=re.IGNORECASE)

    # Convert mid-sentence commas to EIT pause notation
    raw = re.sub(r'(?<!\d),(?!\s*\d)', ' ...', raw)
    raw = re.sub(r'\s+', ' ', raw).strip()
    raw = raw.rstrip('.?!¿¡')
    return raw.strip()


# Show example prompts for first 3 sentences
print('Example prompts passed to Whisper:\n')
for i in [0, 9, 19]:  # sentences 1, 10, 20
    print(f'[Sentence {i+1}]')
    print(build_prompt(EIT_VERSION_A_STIMULI[i]))
    print()

In [ ]:
import whisper

# Load model — large-v3 recommended; use 'medium' on CPU-only Colab
MODEL_SIZE = 'large-v3'

print(f'Loading whisper-{MODEL_SIZE}...')
model = whisper.load_model(MODEL_SIZE, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Model loaded on {next(model.parameters()).device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

---
## Section 8 — Single Sentence Demo

Test the pipeline on one sentence before running the full batch. Compare output with vs. without the EIT-specific prompt.

In [ ]:
def trim_to_speech(audio_np, sr=16000, silence_thresh=-40):
    """Trim trailing silence from response segment."""
    if len(audio_np) < sr * 0.1:
        return audio_np
    pcm = (audio_np * 32767).astype(np.int16)
    seg = AudioSegment(pcm.tobytes(), frame_rate=sr, sample_width=2, channels=1)
    ns  = detect_nonsilent(seg, min_silence_len=250, silence_thresh=silence_thresh)
    if not ns:
        return audio_np
    end_s = min(int(ns[-1][1] / 1000 * sr) + int(0.2 * sr), len(audio_np))
    return audio_np[:end_s]


def is_silent(audio_np, threshold=0.006):
    return float(np.sqrt(np.mean(audio_np ** 2))) < threshold


def transcribe_segment(audio_np, sr, stimulus, model):
    """Transcribe one participant response segment."""
    trimmed = trim_to_speech(audio_np, sr)

    if is_silent(trimmed) or len(trimmed)/sr < 0.3:
        return '[no response]', -2.0, 1.0, ''

    result = model.transcribe(
        trimmed,
        language='es',
        task='transcribe',
        initial_prompt=build_prompt(stimulus),
        temperature=0.0,
        word_timestamps=True,
        condition_on_previous_text=False,
        no_speech_threshold=0.6,
        logprob_threshold=-1.5,
    )

    raw   = result['text'].strip()
    clean = format_transcription(raw)
    segs  = result.get('segments', [])
    logp  = float(np.mean([s['avg_logprob']    for s in segs])) if segs else -1.0
    nsp   = float(np.mean([s['no_speech_prob'] for s in segs])) if segs else 0.5

    return clean, logp, nsp, raw


# ── Demo: transcribe sentence #6 (medium difficulty) ────────────────────────
if audio_files:
    demo_idx = 5  # sentence 6: "Dudo que sepa manejar muy bien"
    s, e     = windows[demo_idx]
    segment  = y[s:e]
    stim     = EIT_VERSION_A_STIMULI[demo_idx]

    print(f'Sentence {demo_idx+1}: "{stim}"\n')
    display(Audio(segment, rate=sr))

    # With EIT prompt
    clean, logp, nsp, raw = transcribe_segment(segment, sr, stim, model)
    print(f'With EIT prompt  → "{clean}"')
    print(f'  Raw Whisper    : "{raw}"')
    print(f'  Confidence     : avg_logprob={logp:.3f}  no_speech_prob={nsp:.3f}')

    # Without EIT prompt (baseline)
    result_no_prompt = model.transcribe(segment, language='es', temperature=0.0)
    print(f'\nWithout prompt   → "{result_no_prompt["text"].strip()}"')

---
## Section 9 — Full Transcription Pipeline

Run the complete pipeline on all 4 participants.

In [ ]:
from tqdm.notebook import tqdm

def transcribe_participant(audio_path, model, intro_skip_s=55, n_stimuli=30):
    """Transcribe all 30 responses for one participant. Returns DataFrame."""
    pid      = Path(audio_path).stem
    audio_np, sr = load_audio_np(audio_path)
    print(f'\n📂 {pid}  ({len(audio_np)/sr:.1f}s)')

    windows = find_response_windows(audio_np, sr, intro_skip_s=intro_skip_s)

    rows = []
    for i, (start, end) in enumerate(tqdm(windows, desc=f'  Transcribing', leave=False)):
        stimulus = EIT_VERSION_A_STIMULI[i]
        segment  = audio_np[start:end]

        trimmed  = trim_to_speech(segment, sr)
        if is_silent(trimmed) or len(trimmed)/sr < 0.3:
            rows.append(dict(participant_id=pid, sentence_num=i+1,
                             stimulus=stimulus, transcription='[no response]',
                             raw_whisper='', avg_logprob=-2.0,
                             no_speech_prob=1.0, needs_review=False,
                             duration_s=round(len(trimmed)/sr, 2), notes='no speech'))
            continue

        clean, logp, nsp, raw = transcribe_segment(segment, sr, stimulus, model)

        notes, needs_review = [], False
        if logp < -0.85:  needs_review = True; notes.append('low confidence')
        if nsp  > 0.45:   needs_review = True; notes.append('possible silence')
        if 0 < len(clean.split()) < 2: needs_review = True; notes.append('very short')

        # Flag possible hallucination (output too similar to stimulus)
        sw = set(stimulus.lower().split())
        ow = set(clean.lower().split())
        if sw and len(sw & ow) / len(sw) > 0.90 and len(clean) > 8:
            notes.append('possible hallucination — verify'); needs_review = True

        rows.append(dict(
            participant_id=pid,
            sentence_num=i+1,
            stimulus=stimulus,
            transcription=clean,
            raw_whisper=raw,
            avg_logprob=round(logp, 4),
            no_speech_prob=round(nsp, 4),
            needs_review=needs_review,
            duration_s=round(len(trimmed)/sr, 2),
            notes='; '.join(notes),
        ))

    df = pd.DataFrame(rows)
    n_rev  = df['needs_review'].sum()
    n_nore = (df['transcription'] == '[no response]').sum()
    print(f'  ✅ {len(df)} sentences | {n_rev} flagged | {n_nore} no-response')
    return df


# ── Run all 4 participants ────────────────────────────────────────────────────
results = {}
if audio_files:
    for f in audio_files:
        df = transcribe_participant(str(f), model)
        results[f.stem] = df
    print(f'\n🏁 Transcription complete: {sum(len(d) for d in results.values())} sentences total')
else:
    print('⚠️  No audio files — place mp3 files in ./data/audio/')

---
## Section 10 — Results Preview

In [ ]:
if results:
    for pid, df in results.items():
        print(f'\n{'='*70}')
        print(f'Participant: {pid}')
        print('='*70)
        display_cols = ['sentence_num', 'stimulus', 'transcription', 'needs_review', 'avg_logprob']
        pd.set_option('display.max_colwidth', 55)
        pd.set_option('display.width', 120)
        display(df[display_cols])
        print(f'\nFlagged for review: {df["needs_review"].sum()} / {len(df)}')
        print(f'[no response]     : {(df["transcription"]=="[no response]").sum()}')
        print(f'Mean log prob     : {df["avg_logprob"].mean():.3f}')

---
## Section 11 — Save Outputs

Two files are produced:
1. **Filled template** (`AutoEIT_Sample_Audio_for_Transcribing_FILLED.xlsx`) — matches the format provided by mentors, with transcriptions in column C
2. **Full report** (`transcription_report.xlsx`) — includes all metadata (confidence scores, flags, raw Whisper output)

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment

YELLOW = PatternFill(start_color='FFFF99', end_color='FFFF99', fill_type='solid')
GREEN  = PatternFill(start_color='D4EDDA', end_color='D4EDDA', fill_type='solid')
ORANGE = PatternFill(start_color='FFE5CC', end_color='FFE5CC', fill_type='solid')
HEADER = PatternFill(start_color='2F5496', end_color='2F5496', fill_type='solid')


def match_sheet(pid, sheet_names):
    pid_n = re.sub(r'[^0-9A-Z]', '', pid.upper())
    for s in sheet_names:
        if s == 'Info': continue
        sn = re.sub(r'[^0-9A-Z]', '', s.upper())
        if pid_n in sn or sn in pid_n:
            return s
    return None


def write_filled_template(results, template_path, output_path):
    """Fill transcriptions into the AutoEIT Excel template (column C)."""
    wb = openpyxl.load_workbook(template_path)

    for pid, df in results.items():
        sheet = match_sheet(pid, wb.sheetnames)
        if not sheet:
            print(f'  ⚠️  No sheet for {pid}')
            continue
        ws = wb[sheet]

        # Add notes column header
        h = ws.cell(1, 4)
        h.value = 'Pipeline Notes (confidence)'
        h.fill  = HEADER
        h.font  = Font(color='FFFFFF', bold=True)

        for _, row in df.iterrows():
            er  = int(row['sentence_num']) + 1   # row 1 = header
            ct  = ws.cell(er, 3)                 # Column C = Transcription
            cn  = ws.cell(er, 4)                 # Column D = Notes

            ct.value     = row['transcription']
            ct.alignment = Alignment(wrap_text=True)

            cn.value = (f"{row['notes']} | " if row['notes'] else '') + \
                       f"logp={row['avg_logprob']:.2f} nsp={row['no_speech_prob']:.2f}"
            cn.font  = Font(size=8, italic=True, color='666666')

            if   row['transcription'] == '[no response]': ct.fill = cn.fill = ORANGE
            elif row['needs_review']:                     ct.fill = cn.fill = YELLOW
            else:                                         ct.fill = GREEN

        ws.column_dimensions['C'].width = 55
        ws.column_dimensions['D'].width = 42

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_path)
    print(f'💾 Filled template → {output_path}')


def save_full_report(results, output_path):
    """Save detailed report with all metadata."""
    summary, all_dfs = [], []
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for pid, df in results.items():
            out = df[['sentence_num','stimulus','transcription','needs_review',
                      'avg_logprob','no_speech_prob','duration_s','notes','raw_whisper']].copy()
            out.columns = ['#','Stimulus','Transcription','Review?',
                           'Log Prob','No Speech','Dur(s)','Notes','Raw Whisper']
            out.to_excel(writer, sheet_name=pid[:31], index=False)
            ws = writer.sheets[pid[:31]]
            ws.column_dimensions['B'].width = 48
            ws.column_dimensions['C'].width = 48
            ws.column_dimensions['H'].width = 30
            all_dfs.append(df)
            summary.append({'Participant': pid, 'Sentences': len(df),
                            'No Response': int((df['transcription']=='[no response]').sum()),
                            'Needs Review': int(df['needs_review'].sum()),
                            'Mean Log Prob': round(df['avg_logprob'].mean(), 3)})

        pd.DataFrame(summary).to_excel(writer, sheet_name='Summary', index=False)
        if all_dfs:
            pd.concat(all_dfs, ignore_index=True).to_excel(writer, sheet_name='All', index=False)

    print(f'📊 Full report → {output_path}')


if results and Path(TEMPLATE_XLS).exists():
    OUT_TEMPLATE = OUTPUT_DIR / 'AutoEIT_Sample_Audio_for_Transcribing_FILLED.xlsx'
    OUT_REPORT   = OUTPUT_DIR / 'transcription_report.xlsx'
    write_filled_template(results, TEMPLATE_XLS, str(OUT_TEMPLATE))
    save_full_report(results, str(OUT_REPORT))
elif results:
    OUT_REPORT = OUTPUT_DIR / 'transcription_report.xlsx'
    save_full_report(results, str(OUT_REPORT))
    print('ℹ️  Template not found — only saving full report')

---
## Section 12 — Evaluation Against Human Transcriptions

Compare pipeline output against the human reference transcriptions using **Word Error Rate (WER)**.

**Target:** ≤10% WER = ≥90% agreement with human transcribers (from proposal specification).

In [ ]:
from jiwer import wer, cer

def normalize_for_wer(text):
    """Normalize text for fair WER comparison."""
    if not isinstance(text, str) or text.strip() == '[no response]':
        return ''
    text = text.lower().strip()
    # Strip diacritics for fair comparison
    text = ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')
    # Normalize pauses and punctuation
    text = re.sub(r'\.{2,}', ' PAUSE ', text)
    text = re.sub(r'[^\w\s\-]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def get_transcription_col(df):
    """Extract transcription column regardless of naming."""
    for col in ['Transcription Rater 1', 'Transcription', 'transcription']:
        if col in df.columns:
            return df[col].tolist()
    return df.iloc[:, 2].tolist()  # fallback: column index 2


if results and Path(REFERENCE_XLS).exists():
    ref_xl = pd.ExcelFile(REFERENCE_XLS)
    print('Reference sheets:', ref_xl.sheet_names)

    eval_rows, summary_rows = [], []

    for pid, hyp_df in results.items():
        # Match participant to reference sheet
        pid_n  = re.sub(r'[^0-9A-Z]', '', pid.upper())
        ref_sheet = None
        for s in ref_xl.sheet_names:
            sn = re.sub(r'[^0-9A-Z]', '', s.upper())
            if pid_n in sn or sn in pid_n:
                ref_sheet = s; break

        if not ref_sheet:
            print(f'  ⚠️  No reference sheet for {pid}')
            continue

        ref_df   = ref_xl.parse(ref_sheet)
        hyp_txts = hyp_df['transcription'].tolist()
        ref_txts = get_transcription_col(ref_df)

        for i, (h_raw, r_raw) in enumerate(zip(hyp_txts, ref_txts)):
            h, r = normalize_for_wer(str(h_raw)), normalize_for_wer(str(r_raw))
            no_resp = str(h_raw).strip() == '[no response]'

            if not r or no_resp:
                sw, sc = None, None
            else:
                try:   sw, sc = round(wer(r, h), 4), round(cer(r, h), 4)
                except: sw, sc = None, None

            eval_rows.append({'participant': pid, 'sentence_num': i+1,
                              'hypothesis': h_raw, 'reference': r_raw,
                              'WER': sw, 'CER': sc, 'no_response': no_resp})

    eval_df = pd.DataFrame(eval_rows)
    valid   = eval_df.dropna(subset=['WER', 'CER'])

    print(f'\n{'='*55}')
    print('EVALUATION RESULTS')
    print('='*55)
    print(f'Sentences evaluated : {len(valid)} / {len(eval_df)}')
    print(f'[no response]       : {eval_df["no_response"].sum()}')
    print(f'Mean WER            : {valid["WER"].mean():.1%}')
    print(f'Mean CER            : {valid["CER"].mean():.1%}')
    print(f'Agreement (1-WER)   : {(1-valid["WER"].mean()):.1%}')
    target_met = (1-valid['WER'].mean()) >= 0.90
    print(f'Target (≥90%)       : {"✅ ACHIEVED" if target_met else "⚠️ BELOW TARGET"}')
    print(f'WER < 10%           : {(valid["WER"] < 0.10).sum()} sentences')
    print(f'WER < 30%           : {(valid["WER"] < 0.30).sum()} sentences')
    print(f'WER > 50%           : {(valid["WER"] > 0.50).sum()} sentences')

    print(f'\nPer-participant breakdown:')
    for pid, grp in valid.groupby('participant'):
        print(f'  {pid}: WER={grp["WER"].mean():.1%}  agreement={(1-grp["WER"].mean()):.1%}')

    # Save evaluation report
    eval_out = OUTPUT_DIR / 'evaluation_report.xlsx'
    with pd.ExcelWriter(eval_out, engine='openpyxl') as writer:
        eval_df.to_excel(writer, sheet_name='All Sentences', index=False)
        valid.groupby('participant')[['WER','CER']].mean().round(4).to_excel(writer, sheet_name='Summary')
    print(f'\n💾 Evaluation report → {eval_out}')

else:
    print('ℹ️  Reference file not found — skipping WER evaluation')
    print(f'   Expected: {REFERENCE_XLS}')
    eval_df = None

---
## Section 13 — Analysis & Visualisation

In [ ]:
if results:
    all_df = pd.concat(results.values(), ignore_index=True)

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle('AutoEIT Pipeline Analysis', fontsize=13, fontweight='bold')

    # 1. Confidence distribution
    axes[0,0].hist(all_df['avg_logprob'].dropna(), bins=20,
                   color='#3498db', edgecolor='white', alpha=0.85)
    axes[0,0].axvline(-0.85, color='#e74c3c', lw=1.5, linestyle='--',
                      label='Review threshold')
    axes[0,0].set_title('Confidence Distribution', fontsize=10)
    axes[0,0].set_xlabel('avg_logprob (higher = more confident)')
    axes[0,0].set_ylabel('Count')
    axes[0,0].legend(fontsize=8)

    # 2. Review flags by participant
    rev_by_p = all_df.groupby('participant_id')['needs_review'].sum()
    rev_by_p.plot(kind='bar', ax=axes[0,1], color='#e74c3c', edgecolor='white', alpha=0.85)
    axes[0,1].set_title('Sentences Flagged for Review', fontsize=10)
    axes[0,1].set_ylabel('Count')
    axes[0,1].tick_params(axis='x', rotation=20)

    # 3. Transcription word count vs sentence complexity
    wc = all_df['transcription'].apply(lambda x: len(x.split()) if x != '[no response]' else 0)
    axes[0,2].scatter(all_df['sentence_num'], wc,
                      c=all_df['avg_logprob'], cmap='RdYlGn',
                      alpha=0.6, s=30, vmin=-2, vmax=0)
    axes[0,2].set_title('Response Length vs Sentence #\n(color = confidence)', fontsize=10)
    axes[0,2].set_xlabel('Sentence #')
    axes[0,2].set_ylabel('Word count in transcription')

    # 4. WER by sentence # (if eval available)
    if 'eval_df' in dir() and eval_df is not None and 'WER' in eval_df.columns:
        valid_eval = eval_df.dropna(subset=['WER'])
        if len(valid_eval) > 0:
            wer_by_sent = valid_eval.groupby('sentence_num')['WER'].mean()
            colors_wer  = ['#e74c3c' if w > 0.5 else '#f39c12' if w > 0.1 else '#27ae60'
                           for w in wer_by_sent]
            axes[1,0].bar(wer_by_sent.index, wer_by_sent.values,
                          color=colors_wer, edgecolor='none')
            axes[1,0].axhline(0.10, color='black', lw=1, linestyle='--',
                              label='10% WER target')
            axes[1,0].set_title('Mean WER by Sentence Number', fontsize=10)
            axes[1,0].set_xlabel('Sentence #')
            axes[1,0].set_ylabel('WER')
            axes[1,0].legend(fontsize=8)

            # 5. WER histogram
            axes[1,1].hist(valid_eval['WER'].dropna(), bins=20,
                           color='#9b59b6', edgecolor='white', alpha=0.85)
            axes[1,1].axvline(0.10, color='#27ae60', lw=1.5, linestyle='--',
                              label='10% threshold')
            axes[1,1].axvline(valid_eval['WER'].mean(), color='#e74c3c',
                              lw=1.5, linestyle='-',
                              label=f'Mean {valid_eval["WER"].mean():.1%}')
            axes[1,1].set_title('WER Distribution', fontsize=10)
            axes[1,1].set_xlabel('WER')
            axes[1,1].set_ylabel('Count')
            axes[1,1].legend(fontsize=8)

            # 6. WER vs sentence complexity (syllables)
            if len(valid_eval) == len(syllable_counts) * len(results):
                # Multiple participants — average per sentence
                avg_wer = valid_eval.groupby('sentence_num')['WER'].mean().values
                axes[1,2].scatter(syllable_counts, avg_wer,
                                  c='#e67e22', alpha=0.8, s=50)
                axes[1,2].set_title('WER vs Sentence Length (syllables)', fontsize=10)
                axes[1,2].set_xlabel('Sentence length (syllables)')
                axes[1,2].set_ylabel('Mean WER')
        else:
            axes[1,0].text(0.5, 0.5, 'No WER data', ha='center', va='center',
                           transform=axes[1,0].transAxes)
    else:
        for ax in axes[1, :]:
            ax.text(0.5, 0.5, 'Run evaluation cell first\n(Section 12)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10,
                    color='gray')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'full_analysis.png', dpi=130, bbox_inches='tight')
    plt.show()

---
## Section 14 — Qualitative Analysis: Difficult Sentences

In [ ]:
if results:
    all_df = pd.concat(results.values(), ignore_index=True)

    # High complexity sentences (16–18 syllables) — sentences 20-30
    hard = all_df[all_df['sentence_num'] >= 20].copy()
    print('=== Long/Complex Sentences (20–30) ===')
    for _, row in hard.head(10).iterrows():
        flag = '⚠️ ' if row['needs_review'] else '   '
        print(f'{flag}[{row["participant_id"]} S{row["sentence_num"]:02d}]')
        print(f'     Target     : {row["stimulus"]}')
        print(f'     Transcript : {row["transcription"]}')
        if row['notes']:
            print(f'     Notes      : {row["notes"]}')
        print()

    # Flagged sentences for manual review
    flagged = all_df[all_df['needs_review']]
    if len(flagged) > 0:
        print(f'\n=== Sentences Flagged for Human Review ({len(flagged)}) ===')
        for _, row in flagged.iterrows():
            print(f'[{row["participant_id"]} S{row["sentence_num"]:02d}] logp={row["avg_logprob"]:.2f}')
            print(f'     Target : {row["stimulus"]}')
            print(f'     Output : {row["transcription"]}')
            print(f'     Notes  : {row["notes"]}')
            print()

---
## Section 15 — Summary & Discussion

### Pipeline Design Decisions

**1. Tone detection as the primary segmentation strategy**

The biggest challenge is that participant recordings contain both the EIT playback (the stimulus sentences) and the participant's voice. A naive Whisper transcription of the full audio would produce the correct target sentences verbatim — not the learner's actual production. The EIT protocol's tone beep (~880Hz) provides a clean temporal boundary. Tone detection at this frequency using a narrow bandpass filter correctly identifies response windows without relying on silence thresholds that might be confused by the stimulus playback.

**2. Stimulus-aware `initial_prompt`**

Whisper's `initial_prompt` biases the decoder by providing prior context. By including the target sentence and explicit instructions to preserve disfluencies, we reduce two failure modes:
- *Under-transcription*: Whisper omitting disfluencies/errors it would normally clean
- *Hallucination*: Whisper generating the target sentence rather than what was actually said

**3. No grammar correction**

This is the most critical constraint. The EIT scoring rubric is meaning-based (0–4). A learner producing `*ella terminó pintar su apartamento*` (incorrect: missing `ha` and `de`) still preserves full meaning → scores 3. If the pipeline auto-corrects this to `ella ha terminado de pintar su apartamento`, the rater would incorrectly assign a 4. Every post-processing step is designed around this constraint.

---

### Known Limitations

1. **Tone frequency assumption**: Set to 880Hz (A5). If the actual NIU recordings use a different frequency, detection will fall back to silence-based segmentation. Easy fix: scan multiple frequencies and use the one with closest to 30 detections.

2. **Heavy L1 transfer**: For learners with phonological systems distant from Spanish (e.g. Mandarin tones, Arabic pharyngeals), Whisper may substitute phonetically similar Spanish words. A fine-tuned model on corrected EIT data would handle this better.

3. **Hallucination on silence**: When a participant responds with silence, Whisper sometimes produces plausible Spanish text. The `is_silent()` check and `no_speech_prob` flag catch most of these.

---

### GSoC Project Plan

If selected, I plan to extend this baseline into a production pipeline:

| Phase | Work |
|---|---|
| **1. Robust segmentation** | Forced-alignment using known target sentences + WebMAUS/Praat; stereo channel separation |
| **2. Fine-tuning** | Active learning loop: pipeline flags uncertain sentences → mentor corrections → Whisper fine-tune |
| **3. Scoring module** | Prompt-based scoring using an LLM (Claude/GPT-4) guided by the Ortega rubric |
| **4. API + UI** | FastAPI endpoint + Gradio interface for lab deployment |
| **5. Evaluation** | Inter-rater reliability comparison: pipeline vs. two human raters (target: ICC ≥ 0.95) |

---

### References

- Ortega, L., Iwashita, N., Norris, J.M., & Rabie, S. (2002). An investigation of elicited imitation tasks in crosslinguistic SLA research. *2nd Language Research Forum.*
- Faretta-Stutenberg, M., Issa, B., Bowden, H.W., & Morgan-Short, K. (2023). Parallel forms reliability of two versions of the Spanish EIT. *Research Methods in Applied Linguistics, 2*(3). https://doi.org/10.1016/j.rmal.2023.100070
- Radford, A. et al. (2023). Robust speech recognition via large-scale weak supervision. *ICML 2023.*